### **Lexi-Guard 모델 학습**

In [ ]:
!pip uninstall -y torchvision torchaudio torch

!pip install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 -q
!pip install transformers==4.52.4 datasets==3.6.0 accelerate==1.7.0 scikit-learn -q

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Drive에 데이터 올려두는 경로 (본인 경로로 수정)
DRIVE_PATH = "/content/drive/MyDrive/lexi-guard"

import os
os.makedirs(f"{DRIVE_PATH}/splits", exist_ok=True)

# 로컬에서 splits/ 폴더를 Drive에 올려두거나
# 아래처럼 직접 업로드
from google.colab import files
# files.upload()  # 필요 시 주석 해제

Mounted at /content/drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json, numpy as np, torch
from torch import nn
from pathlib import Path
from collections import defaultdict
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
from datasets import Dataset

MODEL_NAME = "klue/roberta-large"
MAX_LENGTH = 512
BATCH_SIZE = 8      # OOM 나면 4로
EPOCHS     = 5
LR         = 2e-5
OUTPUT_DIR = f"{DRIVE_PATH}/lexi-guard-roberta"  # Drive에 저장
# OUTPUT_DIR = "/content/lexi-guard-roberta"
LABEL2ID   = {"Faithful": 0, "Not_Faithful": 1}
ID2LABEL   = {0: "Faithful", 1: "Not_Faithful"}

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")

GPU: Tesla T4


In [ ]:
# 데이터 불러오기
def load_split(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

SPLIT_DIR = f"{DRIVE_PATH}/splits"
train_data = load_split(f"{SPLIT_DIR}/train.json")
val_data   = load_split(f"{SPLIT_DIR}/val.json")
test_data  = load_split(f"{SPLIT_DIR}/test.json")

print(f"Train: {len(train_data)}개")
print(f"Val:   {len(val_data)}개")
print(f"Test:  {len(test_data)}개")

Train: 1285개
Val:   151개
Test:  170개


In [ ]:
# 전처리 + 토크나이저
def make_records(data):
    return [
        {
            "text_a": d["context"],
            "text_b": d["answer"],
            "label":  LABEL2ID[d["label_gt"]],
        }
        for d in data
    ]

train_records = make_records(train_data)
val_records   = make_records(val_data)
test_records  = make_records(test_data)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text_a"],
        batch["text_b"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

train_dataset = Dataset.from_list(train_records).map(tokenize, batched=True)
val_dataset   = Dataset.from_list(val_records).map(tokenize, batched=True)
test_dataset  = Dataset.from_list(test_records).map(tokenize, batched=True)

for ds in [train_dataset, val_dataset, test_dataset]:
    ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print("토크나이징 완료")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Map:   0%|          | 0/1285 [00:00<?, ? examples/s]

Map:   0%|          | 0/151 [00:00<?, ? examples/s]

Map:   0%|          | 0/170 [00:00<?, ? examples/s]

토크나이징 완료


In [ ]:
# 클래스 가중치 + 커스텀 Trainer
labels_train = [d["label"] for d in train_records]
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=labels_train,
)
class_weights = torch.tensor(class_weights, dtype=torch.float)
print(f"Class weights — Faithful: {class_weights[0]:.3f}, Not_Faithful: {class_weights[1]:.3f}")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(
            weight=class_weights.to(outputs.logits.device)
        )(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "f1":              f1_score(labels, preds, average="macro"),
        "precision":       precision_score(labels, preds, average="macro"),
        "recall":          recall_score(labels, preds, average="macro"),
        "f1_not_faithful": f1_score(labels, preds, pos_label=1),
    }

Class weights — Faithful: 3.060, Not_Faithful: 0.598


In [ ]:
# 모델 + 학습
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=True,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at klue/roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,F1,Precision,Recall,F1 Not Faithful
1,0.464000,0.201487,0.861213,0.820513,0.944444,0.941176
2,0.148000,0.084152,0.954898,0.931034,0.984127,0.983871
3,0.069900,0.088525,0.954898,0.931034,0.984127,0.983871
4,0.079100,0.122843,0.944424,0.916667,0.980159,0.979757


TrainOutput(global_step=644, training_loss=0.27626067881258376, metrics={'train_runtime': 616.9964, 'train_samples_per_second': 10.413, 'train_steps_per_second': 1.305, 'total_flos': 4790127207505920.0, 'train_loss': 0.27626067881258376, 'epoch': 4.0})

In [ ]:
# 테스트셋 평가
print("===== Test Set 평가 =====")
predictions = trainer.predict(test_dataset)
preds  = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print(classification_report(
    labels, preds,
    target_names=["Faithful", "Not_Faithful"]
))

===== Test Set 평가 =====


              precision    recall  f1-score   support

    Faithful       0.90      1.00      0.95        28
Not_Faithful       1.00      0.98      0.99       142

    accuracy                           0.98       170
   macro avg       0.95      0.99      0.97       170
weighted avg       0.98      0.98      0.98       170



In [ ]:
# 유형별 Error Analysis
print("===== 유형별 F1 (Not_Faithful 기준) =====")

type_preds  = defaultdict(list)
type_labels = defaultdict(list)

for i, d in enumerate(test_data):
    h_type = d.get("hallucination_type", "Faithful")
    type_preds[h_type].append(int(preds[i]))
    type_labels[h_type].append(int(labels[i]))

for h_type in sorted(type_preds.keys()):
    tp = type_labels[h_type]
    pp = type_preds[h_type]
    if len(set(tp)) < 2:
        # Faithful만 있는 경우 acc로 대체
        acc = sum(t == p for t, p in zip(tp, pp)) / len(tp)
        print(f"  {h_type}: Acc={acc:.3f} (n={len(tp)}, Faithful only)")
        continue
    f1 = f1_score(tp, pp, average="binary", pos_label=1)
    print(f"  {h_type}: F1={f1:.3f} (n={len(tp)})")

===== 유형별 F1 (Not_Faithful 기준) =====
  Condition_Addition: Acc=0.920 (n=25, Faithful only)
  Condition_Deletion: Acc=1.000 (n=7, Faithful only)
  Entity_Substitution: Acc=1.000 (n=24, Faithful only)
  Information_Omission: Acc=1.000 (n=25, Faithful only)
  Legal_Effect_Reversal: Acc=1.000 (n=25, Faithful only)
  No_Hallucination: Acc=1.000 (n=28, Faithful only)
  Number_Manipulation: Acc=0.909 (n=11, Faithful only)
  Scope_Manipulation: Acc=1.000 (n=25, Faithful only)


In [ ]:
# 상세 Error Analysis
from collections import defaultdict

print("=" * 70)
print("Error Analysis")
print("=" * 70)

type_result = defaultdict(list)

for i, sample in enumerate(test_data):

    h_type = sample.get("hallucination_type_gt") or \
             sample.get("hallucination_type", "No_Hallucination")

    correct = (preds[i] == labels[i])

    type_result[h_type].append({
        "correct": correct,
        "gt": labels[i],
        "pred": preds[i],
        "context": sample["context"],
        "answer": sample["answer"]
    })

for h_type in sorted(type_result.keys()):

    total = len(type_result[h_type])
    correct = sum(x["correct"] for x in type_result[h_type])
    wrong = total - correct
    acc = correct / total

    print("\n" + "="*10)
    print(f"{h_type}")
    print("="*10)
    print(f"Accuracy : {acc:.3f}")
    print(f"Correct  : {correct}")
    print(f"Wrong    : {wrong}")
    print(f"Total    : {total}")

    if wrong > 0:
        print("\nWrong Samples\n")

        cnt = 1

        for x in type_result[h_type]:
            if not x["correct"]:

                print(f"[{cnt}]")
                print(f"GT   : {ID2LABEL[x['gt']]}")
                print(f"PRED : {ID2LABEL[x['pred']]}")
                print()

                print("Context")
                print(x["context"])
                print()

                print("Answer")
                print(x["answer"])
                print("-"*60)

                cnt += 1

print("\n")
print("="*70)
print("Overall Summary")
print("="*70)

for h_type in sorted(type_result.keys()):
    total = len(type_result[h_type])
    correct = sum(x["correct"] for x in type_result[h_type])
    acc = correct / total
    print(f"{h_type:25s} : {acc:.3f} ({correct}/{total})")

Error Analysis

Condition_Addition
Accuracy : 0.920
Correct  : 23
Wrong    : 2
Total    : 25

Wrong Samples

[1]
GT   : Not_Faithful
PRED : Faithful

Context
제52조(선택적 근로시간제) ① ① 사용자는 취업규칙(취업규칙에 준하는 것을 포함한다)에 따라 업무의 시작 및 종료 시각을 근로자의 결정에 맡기기로 한 근로자에 대하여 근로자대표와의 서면 합의에 따라 다음 각 호의 사항을 정하면 1개월(신상품 또는 신기술의 연구개발 업무의 경우에는 3개월로 한다) 이내의 정산기간을 평균하여 1주간의 근로시간이 제50조제1항의 근로시간을 초과하지 아니하는 범위에서 1주 간에 제50조제1항의 근로시간을, 1일에 제50조제2항의 근로시간을 초과하여 근로하게 할 수 있다. <개정 2021.1.5> 1.  대상 근로자의 범위(15세 이상 18세 미만의 근로자는 제외한다) 2.  정산기간 3.  정산기간의 총 근로시간 4.  반드시 근로하여야 할 시간대를 정하는 경우에는 그 시작 및 종료 시각 5.  근로자가 그의 결정에 따라 근로할 수 있는 시간대를 정하는 경우에는 그 시작 및 종료 시각 6.  그 밖에 대통령령으로 정하는 사항 ② ② 사용자는 제1항에 따라 1개월을 초과하는 정산기간을 정하는 경우에는 다음 각 호의 조치를 하여야 한다. <신설 2021.1.5> 1.  근로일 종료 후 다음 근로일 시작 전까지 근로자에게 연속하여 11시간 이상의 휴식 시간을 줄 것. 다만, 천재지변 등 대통령령으로 정하는 불가피한 경우에는 근로자대표와의 서면 합의가 있으면 이에 따른다. 2.  매 1개월마다 평균하여 1주간의 근로시간이 제50조제1항의 근로시간을 초과한 시간에 대해서는 통상임금의 100분의 50 이상을 가산하여 근로자에게 지급할 것. 이 경우 제56조제1항은 적용하지 아니한다.

Answer
제52조(선택적 근로시간제) ① 사용자는 취업규칙(취업규칙에 준하는 

In [ ]:
# 모델 저장
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"모델 저장 완료: {OUTPUT_DIR}")

모델 저장 완료: /content/drive/MyDrive/lexi-guard/lexi-guard-roberta


In [ ]:
for d in test_data[:5]:
    print(d)

{'id': '5ded8ebeb31477602890a382d9c0ee3c', 'law_name': '고용보험법', 'article_no': '91', 'article_title': '청구의 방식', 'context': '제91조(청구의 방식) 심사의 청구는 대통령령으로 정하는 바에 따라 문서로 하여야 한다.', 'answer': '제91조(청구의 방식) 심사의 청구는 대통령령으로 정하는 바에 따라 문서로 하여야 한다. 다만, 상시근로자 50인 이상 사업장에 한한다.', 'hallucination_type': 'Condition_Addition', 'severity': 'medium', 'subtlety_score': 3, 'severity_subtlety_mismatch': False, 'label_gt': 'Not_Faithful', 'changed_span': {'before': '', 'after': '다만, 상시근로자 50인 이상 사업장에 한한다.'}, 'change_description': '유형 A - 조건 적용', 'plausibility_score': 3, 'judge_reason': "법령 문체는 유지되었으나, '상시근로자 50인 이상 사업장에 한한다'는 조건이 행정심판 절차인 제91조의 성격과 어울리지 않아 법률 전문가 입장에서는 즉각적인 위화감을 느낌. 다만 문장 자체의 구성은 자연스러워 일반인은 법령의 일부로 오인할 가능성이 있음.", 'needs_revalidation': False, 'source': 'llm_generated'}
{'id': '811c2aa0d69b79ada0759af3117a5564', 'law_name': '고용보험법', 'article_no': '91', 'article_title': '청구의 방식', 'context': '제91조(청구의 방식) 심사의 청구는 대통령령으로 정하는 바에 따라 문서로 하여야 한다.', 'answer': '제91조(청구의 방식) 심사의 청구는 대통령령으로 정한다.', 'hallucin

### **KLUE-NLI**

1. 토큰 길이 확인

In [ ]:
import json
import numpy as np
from transformers import AutoTokenizer
from google.colab import drive

drive.mount('/content/drive')

DRIVE_PATH = "/content/drive/MyDrive/lexi-guard"

with open(f"{DRIVE_PATH}/splits/test.json", encoding="utf-8") as f:
    test_data = json.load(f)

nli_tokenizer = AutoTokenizer.from_pretrained(
    "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
)

lengths_ctx = [
    len(nli_tokenizer(d["context"], "")["input_ids"])
    for d in test_data
]

print("CONTEXT ONLY mean:", np.mean(lengths_ctx))
print(f"max: {max(lengths_ctx)}, 95th: {int(np.percentile(lengths_ctx, 95))}, mean: {np.mean(lengths_ctx):.0f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (647 > 512). Running this sequence through the model will result in indexing errors


CONTEXT ONLY mean: 286.01176470588234
max: 1161, 95th: 796, mean: 286


2. 결과 보고 max_length 확정 후 본 추론

In [ ]:
# 셀 1 결과 보고 MAX_LENGTH 숫자만 바꾸면 됨
MAX_LENGTH = 512  # 셀 1 95th percentile 보고 결정

import torch
from sklearn.metrics import (
    classification_report, f1_score,
    accuracy_score, precision_score, recall_score
)
from collections import defaultdict
from transformers import AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LABEL2ID = {"Faithful": 0, "Not_Faithful": 1}

nli_model = AutoModelForSequenceClassification.from_pretrained("MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli")
nli_model.to(device)
nli_model.eval()

label_map = nli_model.config.id2label
print("NLI label map:", label_map)

entail_idx = [k for k, v in label_map.items() if v.lower() == "entailment"][0]
if "neutral" not in [v.lower() for v in label_map.values()]:
    print("Warning: no neutral class")

nli_preds  = []
nli_labels = []

for d in test_data:
    inputs = nli_tokenizer(
        d["context"],
        d["answer"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        return_tensors="pt",
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = nli_model(**inputs).logits

    probs    = torch.softmax(logits, dim=-1)[0]
    pred_nli = torch.argmax(probs).item()
    pred     = 0 if pred_nli == entail_idx else 1
    # entailment → Faithful(0), neutral/contradiction → Not_Faithful(1)

    nli_preds.append(pred)
    nli_labels.append(LABEL2ID[d["label_gt"]])

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

NLI label map: {0: 'entailment', 1: 'neutral', 2: 'contradiction'}


3. 평가

In [ ]:
print("\n===== KLUE-NLI (zero-shot) =====")
print(classification_report(
    nli_labels, nli_preds,
    target_names=["Faithful", "Not_Faithful"],
))

nli_acc  = accuracy_score(nli_labels,  nli_preds)
nli_f1   = f1_score(nli_labels,        nli_preds, average="macro")
nli_prec = precision_score(nli_labels, nli_preds, average="macro")
nli_rec  = recall_score(nli_labels,    nli_preds, average="macro")
f1_faithful     = f1_score(nli_labels, nli_preds, pos_label=0, average="binary")
f1_not_faithful = f1_score(nli_labels, nli_preds, pos_label=1, average="binary")

print(f"Accuracy:        {nli_acc:.4f}")
print(f"Macro F1:        {nli_f1:.4f}")
print(f"Precision:       {nli_prec:.4f}")
print(f"Recall:          {nli_rec:.4f}")
print(f"F1 Faithful:     {f1_faithful:.4f}")
print(f"F1 Not_Faithful: {f1_not_faithful:.4f}")

type_preds_map  = defaultdict(list)
type_labels_map = defaultdict(list)

for i, d in enumerate(test_data):
    h_type = d.get("hallucination_type_gt") or d.get("hallucination_type", "No_Hallucination")
    type_preds_map[h_type].append(nli_preds[i])
    type_labels_map[h_type].append(nli_labels[i])

print("\n[유형별 Not_Faithful F1]")
for h_type in sorted(type_preds_map.keys()):
    preds_t  = type_preds_map[h_type]
    labels_t = type_labels_map[h_type]
    n = len(labels_t)
    if len(set(labels_t)) < 2:
        only = "Faithful" if labels_t[0] == 0 else "Not_Faithful"
        acc  = sum(p == l for p, l in zip(preds_t, labels_t)) / n
        print(f"  {h_type}: Acc={acc:.3f} (n={n}, {only} only)")
    else:
        f1 = f1_score(labels_t, preds_t, pos_label=1, average="binary")
        print(f"  {h_type}: F1={f1:.3f} (n={n})")


===== KLUE-NLI (zero-shot) =====
              precision    recall  f1-score   support

    Faithful       0.38      1.00      0.55        28
Not_Faithful       1.00      0.68      0.81       142

    accuracy                           0.73       170
   macro avg       0.69      0.84      0.68       170
weighted avg       0.90      0.73      0.76       170

Accuracy:        0.7294
Macro F1:        0.6779
Precision:       0.6892
Recall:          0.8380
F1 Faithful:     0.5490
F1 Not_Faithful: 0.8067

[유형별 Not_Faithful F1]
  Condition_Addition: Acc=0.760 (n=25, Not_Faithful only)
  Condition_Deletion: Acc=0.143 (n=7, Not_Faithful only)
  Entity_Substitution: Acc=0.583 (n=24, Not_Faithful only)
  Information_Omission: Acc=0.240 (n=25, Not_Faithful only)
  Legal_Effect_Reversal: Acc=0.960 (n=25, Not_Faithful only)
  No_Hallucination: Acc=1.000 (n=28, Faithful only)
  Number_Manipulation: Acc=0.909 (n=11, Not_Faithful only)
  Scope_Manipulation: Acc=0.880 (n=25, Not_Faithful only)


### **Short vs Long 평가 코드**

1. KLUE-NLI 모델 로드

In [ ]:
import torch
import json
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

NLI_MODEL = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL)
nli_model.to(device)
nli_model.eval()

model.safetensors:   0%|          | 0.00/870M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 1024, padding_idx=0)
      (LayerNorm): LayerNorm((1024,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-23): 24 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (key_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (value_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNo

2. 데이터 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = "/content/drive/MyDrive/lexi-guard"

with open(f"{DRIVE_PATH}/splits/test.json", encoding="utf-8") as f:
    test_data = json.load(f)

LABEL2ID = {"Faithful": 0, "Not_Faithful": 1}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


3. length + short/long 나누기

In [ ]:
lengths = [
    len(nli_tokenizer(d["context"], d["answer"])["input_ids"])
    for d in test_data
]

short_idx = [i for i, l in enumerate(lengths) if l <= 512]
long_idx  = [i for i, l in enumerate(lengths) if l > 512]

4. 모델 inferece (한번 돌리기)

In [ ]:
nli_preds = []
nli_labels = []

for d in test_data:
    inputs = nli_tokenizer(
        d["context"],
        d["answer"],
        truncation=True,
        max_length=512,
        padding="max_length",
        return_tensors="pt"
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = nli_model(**inputs).logits

    probs = torch.softmax(logits, dim=-1)[0]

    entail_idx = [k for k,v in nli_model.config.id2label.items() if "entail" in v.lower()][0]
    contra_idx = [k for k,v in nli_model.config.id2label.items() if "contradict" in v.lower()][0]

    pred = 0 if probs[entail_idx] > probs[contra_idx] else 1

    nli_preds.append(pred)
    nli_labels.append(LABEL2ID[d["label_gt"]])

5. short vs long 평가

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

def eval_subset(idx, name):
    y_true = [nli_labels[i] for i in idx]
    y_pred = [nli_preds[i] for i in idx]

    print(f"\n===== {name} =====")
    print("Acc :", accuracy_score(y_true, y_pred))
    print("F1  :", f1_score(y_true, y_pred, average="macro"))
    print("Prec:", precision_score(y_true, y_pred, average="macro"))
    print("Rec :", recall_score(y_true, y_pred, average="macro"))

eval_subset(short_idx, "SHORT (<=512)")
eval_subset(long_idx, "LONG (>512)")


===== SHORT (<=512) =====
Acc : 0.5210084033613446
F1  : 0.5040579074358411
Prec: 0.6298701298701299
Rec : 0.7121212121212122

===== LONG (>512) =====
Acc : 0.5098039215686274
F1  : 0.49020391843262695
Prec: 0.6212121212121212
Rec : 0.7093023255813954


### **Long vs Short 틀린 샘플 뽑기 코드**

1. 공통 세팅

In [ ]:
lengths = [
    len(nli_tokenizer(d["context"], d["answer"])["input_ids"])
    for d in test_data
]

2. TF-IDF 기준 erro analysis

In [ ]:
short_wrong = []
long_wrong = []

for i, d in enumerate(test_data):
    is_wrong = (nli_preds[i] != nli_labels[i])

    if not is_wrong:
        continue

    if lengths[i] <= 512:
        short_wrong.append(d)
    else:
        long_wrong.append(d)

print("SHORT wrong:", len(short_wrong))
print("LONG wrong:", len(long_wrong))

SHORT wrong: 57
LONG wrong: 25


3. 샘플 10개씩 보기

In [ ]:
print("\n===== SHORT WRONG SAMPLES =====")
for d in short_wrong[:3]:
    print(d["context"][:200])
    print(d["answer"][:200])
    print(d["hallucination_type"])
    print("-"*50)

print("\n===== LONG WRONG SAMPLES =====")
for d in long_wrong[:3]:
    print(d["context"][:200])
    print(d["answer"][:200])
    print(d["hallucination_type"])
    print("-"*50)


===== SHORT WRONG SAMPLES =====
제91조(청구의 방식) 심사의 청구는 대통령령으로 정하는 바에 따라 문서로 하여야 한다.
제91조(청구의 방식) 심사의 청구는 대통령령으로 정하는 바에 따라 문서로 하여야 한다. 다만, 상시근로자 50인 이상 사업장에 한한다.
Condition_Addition
--------------------------------------------------
제91조(청구의 방식) 심사의 청구는 대통령령으로 정하는 바에 따라 문서로 하여야 한다.
제91조(청구의 방식) 심사의 청구는 대통령령으로 정한다.
Information_Omission
--------------------------------------------------
제14조(피보험자격의 상실일) ① ①근로자인 피보험자는 다음 각 호의 어느 하나에 해당하는 날에 각각 그 피보험자격을 상실한다. <개정 2011.7.21, 2019.1.15, 2021.1.5> 1.  근로자인 피보험자가 제10조 및 제10조의2에 따른 적용 제외 근로자에 해당하게 된 경우에는 그 적용 제외 대상자가 된 날 2.  고용산재보험료징수법 제10조
② 자영업자인 피보험자는 고용산재보험료징수법 제49조의2제10항 및 같은 조 제12항에서 준용하는 같은 법 제10조제1호부터 제3호까지의 규정에 보험관계가 소멸한 날에 피보험자격을 상실한다.
Condition_Deletion
--------------------------------------------------

===== LONG WRONG SAMPLES =====
제14조(피보험자격의 상실일) ① ①근로자인 피보험자는 다음 각 호의 어느 하나에 해당하는 날에 각각 그 피보험자격을 상실한다. <개정 2011.7.21, 2019.1.15, 2021.1.5> 1.  근로자인 피보험자가 제10조 및 제10조의2에 따른 적용 제외 근로자에 해당하게 된 경우에는 그 적용 제외 대상자가 된 날 2.  고용산재보험료징수법 제10조
제14조